In [2]:
from pyspark.sql import SparkSession

#Initializes a PySpark session
spark = SparkSession.builder.appName("Restaurant Recommender").getOrCreate()

In [3]:
#Reads the restaurant parquet file
restaurants = spark.read.parquet('restaurants.parquet')
restaurants.show(5)

+--------------------+--------------------+-------------------+------------+-----+-----------+-------------+-------------+-----+------------+-------+--------------------+--------------------+--------------------+
|         business_id|                name|            address|        city|state|postal_code|     latitude|    longitude|stars|review_count|is_open|          attributes|          categories|               hours|
+--------------------+--------------------+-------------------+------------+-----+-----------+-------------+-------------+-----+------------+-------+--------------------+--------------------+--------------------+
|MTSW4McQd7CbVtyjq...|  St Honore Pastries|        935 Race St|Philadelphia|   PA|      19107|   39.9555052|  -75.1555641|  4.0|          80|      1|{'RestaurantsDeli...|Restaurants, Food...|{'Monday': '7:0-2...|
|CF33F8-E6oudUQ46H...|      Sonic Drive-In|      615 S Main St|Ashland City|   TN|      37015|    36.269593|   -87.058943|  2.0|           6|      1

In [4]:
#Reads the ratings parquet file
ratings = spark.read.parquet('reviews.parquet')
ratings.show(5)

+--------------------+--------------------+--------------------+-----+------+-----+----+--------------------+-------------------+
|           review_id|             user_id|         business_id|stars|useful|funny|cool|                text|               date|
+--------------------+--------------------+--------------------+-----+------+-----+----+--------------------+-------------------+
|KU_O5udG6zpxOg-Vc...|mh_-eMZ6K5RLWhZyI...|XQfwVwDr-v0ZS3_Cb...|    3|     0|    0|   0|If you decide to ...|2018-07-07 22:09:11|
|saUsX_uimxRlCVr67...|8g_iMtfSiwikVnbP2...|YjUWPpI6HXG530lwP...|    3|     0|    0|   0|Family diner. Had...|2014-02-05 20:30:30|
|AqPFMleE6RsU23_au...|_7bHUi9Uuf5__HHc_...|kxX2SOes4o-D3ZQBk...|    5|     1|    0|   1|Wow!  Yummy, diff...|2015-01-04 00:01:03|
|Sx8TMOWLNuJBWer-0...|bcjbaE6dDog4jkNY9...|e4Vwtrqf-wpJfwesg...|    4|     1|    0|   1|Cute interior and...|2017-01-14 20:54:15|
|JrIxlS1TzJ-iCu79u...|eUta8W_HdHMXPzLBB...|04UD14gamNjLY0IDY...|    1|     1|    2|   1|I 

In [13]:
import pyspark.sql.functions as F
#Selects all user_ids for users with more than 5 ratings
filtered_users = ratings.groupBy('user_id').count().filter('count > 5').select('user_id')

#Selects only ratings by users with more than 5 ratings
filtered_ratings = ratings.join(filtered_users, on = 'user_id', how = 'inner')

In [15]:
#Counts and prints the number of users/ratings after filtering by count
n_users = filtered_users.count()
n_ratings = filtered_ratings.count()

print(f'{n_ratings} unique ratings by {n_users} users')

2615485 unique ratings by 153780 users


In [16]:
#Selects users, items (restaurants), and ratings from the ratings table
edges = filtered_ratings.select(['user_id', 'business_id', 'stars'])
edges.show(5)

+--------------------+--------------------+-----+
|             user_id|         business_id|stars|
+--------------------+--------------------+-----+
|--4AjktZiHowEIBCM...|EtKSTHV5Qx_Q7Aur9...|    4|
|--4AjktZiHowEIBCM...|wUnLSg_GKfEIQ5CQQ...|    4|
|--4AjktZiHowEIBCM...|LNHq9WxfhN2UBNOR2...|    5|
|--4AjktZiHowEIBCM...|wm5mQ4cSpvko9WlCq...|    5|
|--4AjktZiHowEIBCM...|tYn8hGpZiRgJ8cP2F...|    4|
|--4AjktZiHowEIBCM...|sL6fC0P4C-gyL4E5g...|    3|
|--4AjktZiHowEIBCM...|a62d7e_xXeljJcOUH...|    5|
|--4AjktZiHowEIBCM...|hCMqbFJyLczPk_qU3...|    5|
|--4AjktZiHowEIBCM...|OLkS4jfozQpUIUq0K...|    3|
|--4AjktZiHowEIBCM...|7J_7uvHEV94C4TocW...|    4|
|--4AjktZiHowEIBCM...|YZ0YVMdWUik8ukQp1...|    5|
|--4AjktZiHowEIBCM...|eXKblEHP3YJYU1Awz...|    5|
|--4AjktZiHowEIBCM...|Yz0fJyBkUF8VZBvwF...|    5|
|--4AjktZiHowEIBCM...|vuE1iseFrgNPumUEf...|    5|
|--8r3pNaZiG1fN8LC...|xp7IRO4FDLcHkAO59...|    4|
|--8r3pNaZiG1fN8LC...|h-EsSqXjqA6Jy7Oc7...|    4|
|--8r3pNaZiG1fN8LC...|DJYWYpHtknjb4W0NT...|    2|


In [18]:
#Assigns an integer ID to all unique users
edges_distinct = edges.select('user_id').distinct()
edges_distinct.createOrReplaceTempView('edges_users')
user_id = spark.sql('select row_number() over (order by "user_id") as user_id_int, * from edges_users')

#Assigns an integer ID to all unique restaurants
edges_res_distinct = edges.select('business_id').distinct()
edges_res_distinct.createOrReplaceTempView('edges_items')
item_id = spark.sql('select row_number() over (order by "business_id") as item_id_int, * from edges_items')

#Joins each rating with its int user ID and int restaurant ID
new_edges = edges.join(user_id, "user_id", "inner").join(item_id, "business_id", "inner")
new_edges.show(5)

+--------------------+--------------------+-----+-----------+-----------+
|         business_id|             user_id|stars|user_id_int|item_id_int|
+--------------------+--------------------+-----+-----------+-----------+
|YdZS3QkpjgHU2zIJ_...|-073IXD_JkLK8SlRc...|    4|         18|          1|
|BnibjYoTYefJXQ_ZV...|-1awBy86Qgr3aN30_...|    4|         41|          2|
|LnJSsNVZkStgtj86f...|-2p_A5675Eh6gcZIG...|    3|         58|          3|
|L-VNs3YquPGKVsXl2...|-3-GR6zLPQUQXC02x...|    5|         59|          4|
|L-VNs3YquPGKVsXl2...|-3s52C4zL_DHRK0UL...|    4|         64|          4|
+--------------------+--------------------+-----+-----------+-----------+
only showing top 5 rows


In [19]:
#Splits the ratings table randomly, using 80% for recommender training and 20% for testing
 (train, test) = new_edges.randomSplit([0.8, 0.2], seed = 591)

In [20]:
from pyspark.ml.recommendation import ALS

#Initializes an ALS (alternating least squares) recommender model
#implicitPrefs = False means the model will treat the ratings as explicit (1-5), rather than implicit (1 if user rated that restaurant, 0 otherwise)
#coldStartStrategy = 'drop' means that if the model is asked to predict a rating for a user not in the training set, it will instead drop that row from the test set
als = ALS(
    implicitPrefs = False,
    userCol = 'user_id_int',
    itemCol = 'item_id_int',
    ratingCol = 'stars',
    coldStartStrategy = 'drop',
    maxIter = 5,
    seed = 591
  )

#Fits the model and generates rating predictions on the test set
model = als.fit(train)
predict_df = model.transform(test)

predict_df.show(5)

+--------------------+--------------------+-----+-----------+-----------+----------+
|         business_id|             user_id|stars|user_id_int|item_id_int|prediction|
+--------------------+--------------------+-----+-----------+-----------+----------+
|1NvzccBgS0Xg3MWqN...|---UgP94gokyCDuB5...|    5|          1|      22048|  3.904546|
|hLlDzRaDGN-0SlEWk...|---UgP94gokyCDuB5...|    5|          1|       5584| 4.3039374|
|YcvXLSlXX2lQQK1vW...|--8r3pNaZiG1fN8LC...|    5|          3|      48195| 1.8240621|
|7_kXuKFruGrrRauN9...|--Vu3Gux9nPnLcG9y...|    5|          6|      28718| 4.7516284|
|CRrKt_OSlTXhOQ4Vh...|--Vu3Gux9nPnLcG9y...|    5|          6|      36630|  4.835558|
+--------------------+--------------------+-----+-----------+-----------+----------+
only showing top 5 rows


In [21]:
from pyspark.ml.evaluation import RegressionEvaluator

#Yelp reviews are bounded between 1 and 5, while ratings predictions are unbounded
#This code makes any rating prediction below 1 a 1, and any prediction above a 5 becomes 5
clipped_predictions = predict_df.withColumn(
    'clipped_prediction',
    F.when(F.col('prediction') < 1.0, 1.0)\
    .when(F.col('prediction') > 5.0, 5.0)\
    .otherwise(F.col('prediction'))
)

#The first evaluator calculates the RMSE (root mean squared error) between the (clipped) predictions and actual ratings
evaluator = RegressionEvaluator(
    metricName = 'rmse',
    labelCol = 'stars',
    predictionCol = 'clipped_prediction'
)

#Calculates RMSE using the evaluator and displays the result
rmse = evaluator.evaluate(clipped_predictions)
print(f'RMSE = {rmse}')

#The second evaluator calculates the MAE (mean absolute error)
evaluator = RegressionEvaluator(
    metricName = 'mae',
    labelCol = 'stars',
    predictionCol = 'clipped_prediction'
)

#Calculates MAE using the evaluator and displays the result
mae = evaluator.evaluate(clipped_predictions)
print(f'MAE = {mae}')

RMSE = 0.858934357144623
MAE = 1.0434565338275679


In [23]:
model.save('als_pipeline')

In [24]:
import shutil
shutil.make_archive('als_pipeline', 'zip', '/content/als_pipeline')

'/content/als_pipeline.zip'